# nb_fda_01_analyst_explore
**SYNTHETIC DATA · for training only · no OSFI data**

**LIVE in D4: notebooks for analysts.** Default lakehouse: `lh_supervisory`. Each cell is short; together they run in under 2 minutes on F2 once the Spark session is up.

1. SQL in a notebook (`%%sql`)
2. pandas + Data Wrangler
3. Quarter-over-quarter change and a chart
4. Semantic Link: the model's measures from Python ("one definition, every tool")
5. Save the outliers as a table the team can query

## 1. SQL in a notebook
CET1 ratio by institution type for **2026 Q2**: SUM of capital / SUM of RWA (store components, not ratios).

In [ ]:
%%sql
-- SYNTHETIC DATA · for training only · no OSFI data
SELECT i.institution_type,
       COUNT(DISTINCT c.institution_id)        AS institutions,
       ROUND(SUM(c.cet1_capital) / SUM(c.rwa), 4) AS cet1_ratio
FROM fact_capital AS c
JOIN dim_institution AS i ON i.institution_id = c.institution_id
WHERE c.quarter_key = 20262
GROUP BY i.institution_type
ORDER BY cet1_ratio

*Expected:* D-SIB 0.1328 (6) · Other bank 0.1413 (10) · Trust and loan 0.1504 (6).

## 2. pandas + Data Wrangler

In [ ]:
# SYNTHETIC DATA · for training only · no OSFI data
cap = spark.read.table("fact_capital")
inst = spark.read.table("dim_institution")
qtr = spark.read.table("dim_quarter")

pdf = (cap.join(inst, "institution_id").join(qtr, "quarter_key")
          .selectExpr("institution_code", "institution_name", "institution_type", "peer_group",
                      "quarter_key", "quarter_label", "quarter_index", "cet1_capital", "rwa",
                      "cet1_capital / rwa AS cet1_ratio",
                      "tier1_capital / leverage_exposure AS leverage_ratio")
          .toPandas())
pdf.describe()

**Open it in Data Wrangler:** notebook ribbon **Data** (or the toolbar above the output) > **Launch Data Wrangler** > choose `pdf`.
1. Filter: `quarter_key` equals `20263`.
2. Group by and aggregate: `institution_type`, `cet1_ratio` > Mean.
3. Sort by the mean ascending.
4. **Add code to notebook**: Data Wrangler writes the pandas code for you (review it; no magic).

*Say:* the summary statistics and every step become plain, reviewable pandas code.

## 3. Quarter-over-quarter change and a chart

In [ ]:
# SYNTHETIC DATA · for training only · no OSFI data
import matplotlib.pyplot as plt

pdf = pdf.sort_values(["institution_code", "quarter_index"])
pdf["prev_quarter_index"] = pdf.groupby("institution_code")["quarter_index"].shift(1)
pdf["cet1_qoq_pp"] = (pdf["cet1_ratio"] - pdf.groupby("institution_code")["cet1_ratio"].shift(1)) * 100
pdf.loc[pdf["prev_quarter_index"] != pdf["quarter_index"] - 1, "cet1_qoq_pp"] = None   # gap = no comparison

current = int(qtr.where("is_current").first()["quarter_key"])
latest = pdf[pdf["quarter_key"] == current].sort_values("cet1_qoq_pp")
display(latest[["institution_name", "institution_type", "cet1_ratio", "cet1_qoq_pp"]].head(5))

by_type = (pdf[pdf["quarter_index"] > 31 - 12]
           .groupby(["quarter_label", "institution_type"])[["cet1_capital", "rwa"]].sum())
trend = (by_type["cet1_capital"] / by_type["rwa"]).unstack("institution_type") * 100
ax = trend.plot(figsize=(9, 4), marker="o", title="CET1 ratio by institution type, last 12 quarters (SYNTHETIC)")
ax.set_ylabel("CET1 ratio (%)"); ax.set_xlabel(""); ax.grid(alpha=0.3)
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

*Expected (after D2 landing):* largest 2026 Q3 drops: Driftwood Bank −1.41 pp, Stonebridge Bank −0.64 pp, Riverbend Trust and Loan −0.58 pp.

## 4. Semantic Link: one definition, every tool
The same measures the report, Excel and Copilot use, called from Python. No re-implementation of the CET1 formula.

In [ ]:
# SYNTHETIC DATA · for training only · no OSFI data
import sempy.fabric as fabric

measures = fabric.list_measures("sm_fda_supervisory")
display(measures.head(20))

In [ ]:
# SYNTHETIC DATA · for training only · no OSFI data
result = fabric.evaluate_measure(
    "sm_fda_supervisory",
    measure=["CET1 Ratio", "Institutions Reporting"],
    groupby_columns=["dim_quarter[quarter_label]", "dim_institution[institution_type]"],
    filters={"dim_quarter[quarter_label]": ["2026 Q2", "2026 Q3"]},
)
display(result)

*Expected:* 2026 Q2 CET1 Ratio = the `%%sql` numbers above (D-SIB 13.28%, Other bank 14.13%, Trust and loan 15.04%). Same definition, every tool.

## 5. Save the outliers
Institutions whose CET1 ratio fell more than **0.5 pp** quarter over quarter in the current quarter, saved as the lakehouse table `analyst_outliers`.

In [ ]:
# SYNTHETIC DATA · for training only · no OSFI data
outliers = latest[latest["cet1_qoq_pp"] < -0.5][["institution_code", "institution_name", "institution_type",
                                                  "quarter_key", "cet1_ratio", "cet1_qoq_pp"]]
spark.createDataFrame(outliers).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("analyst_outliers")
print(f"analyst_outliers: {len(outliers)} rows")
display(outliers)

*Expected:* **3 rows** after D2 (Driftwood Bank, Stonebridge Bank, Riverbend Trust and Loan); 2 if the late filings are not landed.

## Summary
* `%%sql`, pandas and Data Wrangler for exploration, in one place, on the same Delta tables as the SQL analytics endpoint.
* Semantic Link reads the **governed measures** from `sm_fda_supervisory`, so Python agrees with the report.
* The result is a table (`analyst_outliers`) others can query from SQL, Power BI or Excel.